In [1]:
!pip install atproto pandas networkx tqdm pyarrow

In [2]:
!pip install matplotlib

In [3]:
%pip install atproto


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
from atproto import Client

BSKY_HANDLE = "flena.bsky.social"
BSKY_APP_PASSWORD = "25s3-doqo-rngr-lnma"

client = Client()
client.login(BSKY_HANDLE, BSKY_APP_PASSWORD)

ProfileViewDetailed(did='did:plc:zsdps3mbxwbypep6e33hd5p5', handle='flena.bsky.social', associated=ProfileAssociated(activity_subscription=ProfileAssociatedActivitySubscription(allow_subscriptions='followers', py_type='app.bsky.actor.defs#profileAssociatedActivitySubscription'), chat=None, feedgens=0, germ=None, labeler=False, lists=0, starter_packs=0, py_type='app.bsky.actor.defs#profileAssociated'), avatar='https://cdn.bsky.app/img/avatar/plain/did:plc:zsdps3mbxwbypep6e33hd5p5/bafkreiacmnedzm5gc6w2254chanxadojhkelmnahsokjzvpu35v3bxwgka', banner=None, created_at='2026-06-30T12:53:06.623Z', debug=None, description=None, display_name='', followers_count=0, follows_count=1, indexed_at='2026-06-30T12:53:56.020Z', joined_via_starter_pack=None, labels=[], pinned_post=None, posts_count=0, pronouns=None, status=None, verification=None, viewer=ViewerState(activity_subscription=None, blocked_by=False, blocking=None, blocking_by_list=None, followed_by=None, following=None, known_followers=None, 

In [6]:
# ============================================================
# SERIOUS TICKER WHITELIST FROM NASDAQ TRADER SYMBOL DIRECTORY
# Compatible with Jupyter/VS Code even if current folder is read-only
# ============================================================

import pandas as pd
import re
import os
import tempfile
from pathlib import Path
from io import StringIO

# If requests/certifi are missing, install them first in a separate cell:
# %pip install requests certifi

import requests
import certifi


# ------------------------------------------------------------
# 1. Create a writable output directory
# ------------------------------------------------------------

def get_writable_output_dir():
    candidate_dirs = [
        Path.cwd() / "outputs",
        Path.home() / "bluesky_sna_outputs",
        Path(tempfile.gettempdir()) / "bluesky_sna_outputs",
    ]

    for directory in candidate_dirs:
        try:
            directory.mkdir(parents=True, exist_ok=True)
            test_file = directory / ".write_test"
            test_file.write_text("test")
            test_file.unlink()
            return directory
        except Exception:
            continue

    raise OSError("No writable output directory found.")

OUTPUT_DIR = get_writable_output_dir()

print("Output directory:", OUTPUT_DIR)
print("Writable:", os.access(OUTPUT_DIR, os.W_OK))


# ------------------------------------------------------------
# 2. Nasdaq Trader URLs
# ------------------------------------------------------------

NASDAQ_LISTED_URL = "https://www.nasdaqtrader.com/dynamic/symdir/nasdaqlisted.txt"
OTHER_LISTED_URL = "https://www.nasdaqtrader.com/dynamic/symdir/otherlisted.txt"


# ------------------------------------------------------------
# 3. Robust downloader
# ------------------------------------------------------------

def download_text_file(url):
    """
    Downloads a public text file using requests + certifi.
    This avoids pandas/urllib SSL problems on some Mac/Python installs.
    """
    headers = {
        "User-Agent": "Mozilla/5.0 academic-research-project"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=30,
        verify=certifi.where()
    )

    response.raise_for_status()
    return response.text


def read_pipe_separated_url(url):
    text = download_text_file(url)

    # Remove empty lines
    lines = [line for line in text.splitlines() if line.strip()]

    # Remove final metadata line if present
    lines = [
        line for line in lines
        if not line.startswith("File Creation Time")
    ]

    cleaned_text = "\n".join(lines)

    return pd.read_csv(StringIO(cleaned_text), sep="|")


# ------------------------------------------------------------
# 4. Load Nasdaq-listed securities
# ------------------------------------------------------------

def load_nasdaq_listed():
    df = read_pipe_separated_url(NASDAQ_LISTED_URL)

    df = df[df["Symbol"].notna()].copy()

    df = df.rename(columns={
        "Symbol": "ticker",
        "Security Name": "security_name",
        "Market Category": "market_category",
        "Test Issue": "test_issue",
        "ETF": "etf"
    })

    df["source_file"] = "nasdaqlisted"
    df["exchange_group"] = "NASDAQ"

    return df[[
        "ticker",
        "security_name",
        "market_category",
        "test_issue",
        "etf",
        "source_file",
        "exchange_group"
    ]]


# ------------------------------------------------------------
# 5. Load other exchange-listed securities
# ------------------------------------------------------------

def load_other_listed():
    df = read_pipe_separated_url(OTHER_LISTED_URL)

    df = df[df["ACT Symbol"].notna()].copy()

    df = df.rename(columns={
        "ACT Symbol": "ticker",
        "Security Name": "security_name",
        "Exchange": "exchange_group",
        "Test Issue": "test_issue",
        "ETF": "etf"
    })

    df["market_category"] = None
    df["source_file"] = "otherlisted"

    return df[[
        "ticker",
        "security_name",
        "market_category",
        "test_issue",
        "etf",
        "source_file",
        "exchange_group"
    ]]


# ------------------------------------------------------------
# 6. Build raw ticker universe
# ------------------------------------------------------------

raw_nasdaq = load_nasdaq_listed()
raw_other = load_other_listed()

tickers_raw = pd.concat([raw_nasdaq, raw_other], ignore_index=True)

tickers_raw["ticker"] = tickers_raw["ticker"].astype(str).str.upper().str.strip()
tickers_raw["security_name"] = tickers_raw["security_name"].astype(str)

tickers_raw = tickers_raw.drop_duplicates(subset=["ticker"])
tickers_raw = tickers_raw.sort_values("ticker").reset_index(drop=True)

print("Raw securities:", len(tickers_raw))

display(tickers_raw.head(20))


# ------------------------------------------------------------
# 7. Save raw ticker universe
# ------------------------------------------------------------

raw_path = OUTPUT_DIR / "us_listed_securities_raw.csv"
tickers_raw.to_csv(raw_path, index=False)

print("Saved raw securities to:", raw_path)

Output directory: /Users/bucco/Downloads/outputs
Writable: True
Raw securities: 12931


,ticker,security_name,market_category,test_issue,etf,source_file,exchange_group
0,A,"Agilent Technologies, Inc. Common Stock",None,N,N,otherlisted,N
1,AA,Alcoa Corporation Common Stock,None,N,N,otherlisted,N
2,AAA,Alternative Access First Priority CLO Bond ETF,None,N,Y,otherlisted,P
3,AAAA,Amplius Aggressive Asset Allocation ETF,None,N,Y,otherlisted,Z
4,AAAC,Columbia AAA CLO ETF,None,N,Y,otherlisted,P
5,AAAD,PGIM AAA CLO Aggregate Duration ETF,None,N,Y,otherlisted,P
6,AAAP,Pacer Barings CLO Market Flex ETF,G,N,Y,nasdaqlisted,NASDAQ
7,AAAU,Goldman Sachs Physical Gold ETF Shares,None,N,Y,otherlisted,Z
8,AAC.U,"Ares Acquisition Corporation III Units, each c...",None,N,N,otherlisted,N
9,AACB,Artius II Acquisition Inc. - Class A Ordinary ...,G,N,N,nasdaqlisted,NASDAQ


Saved raw securities to: /Users/bucco/Downloads/outputs/us_listed_securities_raw.csv


In [7]:
# ============================================================
# BUILD CLEAN WHITELISTS:
# 1. stock_like_tickers: stricter equity-like universe
# 2. financial_symbols: broad listed financial instruments universe
# ============================================================

import re

# ------------------------------------------------------------
# 1. Basic cleaning
# ------------------------------------------------------------

tickers_clean = tickers_raw.copy()

tickers_clean["ticker"] = (
    tickers_clean["ticker"]
    .astype(str)
    .str.upper()
    .str.strip()
)

tickers_clean["security_name"] = (
    tickers_clean["security_name"]
    .astype(str)
    .str.strip()
)

tickers_clean["test_issue"] = (
    tickers_clean["test_issue"]
    .astype(str)
    .str.upper()
    .str.strip()
)

tickers_clean["etf"] = (
    tickers_clean["etf"]
    .astype(str)
    .str.upper()
    .str.strip()
)

# Remove test issues
tickers_clean = tickers_clean[tickers_clean["test_issue"] == "N"].copy()

# Keep only ticker formats that can realistically appear as Bluesky cashtags
# Examples: AAPL, BRK.B, BF.A, RDS-A
tickers_clean = tickers_clean[
    tickers_clean["ticker"].str.match(r"^[A-Z0-9][A-Z0-9\.\-]{0,9}$", na=False)
].copy()

tickers_clean = tickers_clean.drop_duplicates(subset=["ticker"])
tickers_clean = tickers_clean.sort_values("ticker").reset_index(drop=True)

print("Clean listed securities:", len(tickers_clean))


# ------------------------------------------------------------
# 2. Broad whitelist: listed financial symbols
# ------------------------------------------------------------
# This is the recommended whitelist for your Bluesky project
# if the target is a large ticker/asset co-mention network.

financial_symbols_df = tickers_clean.copy()

financial_symbols = sorted(financial_symbols_df["ticker"].unique())

financial_symbols_path = OUTPUT_DIR / "us_listed_financial_symbols_whitelist.csv"
financial_symbols_txt_path = OUTPUT_DIR / "us_listed_financial_symbols.txt"

financial_symbols_df.to_csv(financial_symbols_path, index=False)

with open(financial_symbols_txt_path, "w") as f:
    for ticker in financial_symbols:
        f.write(ticker + "\n")

print("Broad financial symbols:", len(financial_symbols))
print("Saved:", financial_symbols_path)
print("Saved:", financial_symbols_txt_path)


# ------------------------------------------------------------
# 3. Stricter stock-like whitelist
# ------------------------------------------------------------
# This tries to remove ETFs, funds, warrants, units, rights,
# preferred shares, notes, bonds, and other non-common-equity instruments.
# It is useful for reporting a cleaner stock-only universe.

EXCLUDE_NAME_PATTERNS = [
    r"\bETF\b",
    r"\bETFS\b",
    r"\bETN\b",
    r"\bETNS\b",
    r"\bFUND\b",
    r"\bTRUST\b",
    r"\bINDEX\b",
    r"\bPORTFOLIO\b",
    r"\bINCOME\b",
    r"\bWARRANT\b",
    r"\bWARRANTS\b",
    r"\bRIGHT\b",
    r"\bRIGHTS\b",
    r"\bUNIT\b",
    r"\bUNITS\b",
    r"\bPREFERRED\b",
    r"\bPREFERENCE\b",
    r"\bPREF\b",
    r"\bDEPOSITARY\b",
    r"\bDEPOSITORY\b",
    r"\bNOTE\b",
    r"\bNOTES\b",
    r"\bBOND\b",
    r"\bBONDS\b",
    r"\bDEBENTURE\b",
    r"\bDEBENTURES\b",
    r"\bSERIES\b",
    r"\bREIT\b",
    r"\bSPAC\b",
    r"\bACQUISITION CORP.*UNIT\b",
    r"\bCLOSED END\b",
    r"\bCLOSED-END\b",
    r"\bBENEFICIAL INTEREST\b",
    r"\bCONVERTIBLE\b",
    r"\bCALLABLE\b",
    r"\bFLOATING RATE\b",
    r"\bSENIOR\b",
    r"\bSUBORDINATED\b",
]

EXCLUDE_TICKER_PATTERNS = [
    r"\.W$",
    r"\.WS$",
    r"\.WT$",
    r"\.U$",
    r"\.R$",
    r"\.RT$",
    r"\.P$",
    r"\.PR",
    r"\.A$",
    r"\.B$",
    r"\.C$",
    r"\.D$",
    r"\.E$",
    r"\.F$",
    r"\.G$",
    r"\.H$",
    r"\.I$",
    r"\.J$",
    r"\.K$",
    r"\.L$",
    r"\.M$",
    r"\.N$",
    r"\.O$",
    r"\.P$",
    r"\.Q$",
    r"\.R$",
    r"\.S$",
    r"\.T$",
    r"\.U$",
    r"\.V$",
    r"\.W$",
    r"\.X$",
    r"\.Y$",
    r"\.Z$",
]

def is_excluded_stock_like(row):
    name = str(row["security_name"]).upper()
    ticker = str(row["ticker"]).upper()

    # remove ETFs directly from Nasdaq field
    if str(row["etf"]).upper() == "Y":
        return True

    for pattern in EXCLUDE_NAME_PATTERNS:
        if re.search(pattern, name, flags=re.IGNORECASE):
            return True

    for pattern in EXCLUDE_TICKER_PATTERNS:
        if re.search(pattern, ticker, flags=re.IGNORECASE):
            return True

    return False

stock_like_df = tickers_clean[
    ~tickers_clean.apply(is_excluded_stock_like, axis=1)
].copy()

stock_like_df = stock_like_df.sort_values("ticker").reset_index(drop=True)

stock_like_tickers = sorted(stock_like_df["ticker"].unique())

stock_like_path = OUTPUT_DIR / "us_stock_like_ticker_whitelist.csv"
stock_like_txt_path = OUTPUT_DIR / "us_stock_like_tickers.txt"

stock_like_df.to_csv(stock_like_path, index=False)

with open(stock_like_txt_path, "w") as f:
    for ticker in stock_like_tickers:
        f.write(ticker + "\n")

print("Stock-like tickers:", len(stock_like_tickers))
print("Saved:", stock_like_path)
print("Saved:", stock_like_txt_path)


# ------------------------------------------------------------
# 4. Display examples
# ------------------------------------------------------------

print("\nExamples from broad financial symbols:")
display(financial_symbols_df.head(20))

print("\nExamples from stricter stock-like symbols:")
display(stock_like_df.head(20))

Clean listed securities: 12511
Broad financial symbols: 12511
Saved: /Users/bucco/Downloads/outputs/us_listed_financial_symbols_whitelist.csv
Saved: /Users/bucco/Downloads/outputs/us_listed_financial_symbols.txt
Stock-like tickers: 5085
Saved: /Users/bucco/Downloads/outputs/us_stock_like_ticker_whitelist.csv
Saved: /Users/bucco/Downloads/outputs/us_stock_like_tickers.txt

Examples from broad financial symbols:


,ticker,security_name,market_category,test_issue,etf,source_file,exchange_group
0,A,"Agilent Technologies, Inc. Common Stock",None,N,N,otherlisted,N
1,AA,Alcoa Corporation Common Stock,None,N,N,otherlisted,N
2,AAA,Alternative Access First Priority CLO Bond ETF,None,N,Y,otherlisted,P
3,AAAA,Amplius Aggressive Asset Allocation ETF,None,N,Y,otherlisted,Z
4,AAAC,Columbia AAA CLO ETF,None,N,Y,otherlisted,P
5,AAAD,PGIM AAA CLO Aggregate Duration ETF,None,N,Y,otherlisted,P
6,AAAP,Pacer Barings CLO Market Flex ETF,G,N,Y,nasdaqlisted,NASDAQ
7,AAAU,Goldman Sachs Physical Gold ETF Shares,None,N,Y,otherlisted,Z
8,AAC.U,"Ares Acquisition Corporation III Units, each c...",None,N,N,otherlisted,N
9,AACB,Artius II Acquisition Inc. - Class A Ordinary ...,G,N,N,nasdaqlisted,NASDAQ



Examples from stricter stock-like symbols:


,ticker,security_name,market_category,test_issue,etf,source_file,exchange_group
0,A,"Agilent Technologies, Inc. Common Stock",None,N,N,otherlisted,N
1,AA,Alcoa Corporation Common Stock,None,N,N,otherlisted,N
2,AACB,Artius II Acquisition Inc. - Class A Ordinary ...,G,N,N,nasdaqlisted,NASDAQ
3,AACI,Armada Acquisition Corp. III - Class A Ordinar...,G,N,N,nasdaqlisted,NASDAQ
4,AACO,Abony Acquisition Corp. I - Class A Ordinary S...,G,N,N,nasdaqlisted,NASDAQ
5,AACP,Apogee Acquisition Corp - Class A Ordinary Shares,G,N,N,nasdaqlisted,NASDAQ
6,AADX,"Applied Aerospace & Defense, Inc. Common Stock",None,N,N,otherlisted,N
7,AAL,"American Airlines Group, Inc. - Common Stock",Q,N,N,nasdaqlisted,NASDAQ
8,AAME,Atlantic American Corporation - Common Stock,G,N,N,nasdaqlisted,NASDAQ
9,AAMI,Acadian Asset Management Inc. Common Stock,None,N,N,otherlisted,N


In [9]:
# ============================================================
# BLUESKY LOGIN
# ============================================================

from atproto import Client

BSKY_HANDLE = "flena.bsky.social"
BSKY_APP_PASSWORD = "tezo-jnci-pdfb-3aia"

client = Client()
profile = client.login(BSKY_HANDLE, BSKY_APP_PASSWORD)

print("Logged in as:", profile.handle)

Logged in as: flena.bsky.social


In [10]:
# ============================================================
# PREPARE VALID TICKER SET FOR BLUESKY EXTRACTION
# ============================================================

VALID_TICKERS = set(financial_symbols)

print("Valid tickers in whitelist:", len(VALID_TICKERS))

# Some tickers contain dots or dashes. Bluesky users may write them differently.
# Example: BRK.B may appear as $BRK.B, $BRK-B, or sometimes $BRKB.
# We create a simple alias map to improve matching.

TICKER_ALIAS_TO_CANONICAL = {}

for ticker in VALID_TICKERS:
    canonical = ticker.upper().strip()
    TICKER_ALIAS_TO_CANONICAL[canonical] = canonical
    
    # Dot/dash variants
    if "." in canonical:
        TICKER_ALIAS_TO_CANONICAL[canonical.replace(".", "-")] = canonical
        TICKER_ALIAS_TO_CANONICAL[canonical.replace(".", "")] = canonical
    
    if "-" in canonical:
        TICKER_ALIAS_TO_CANONICAL[canonical.replace("-", ".")] = canonical
        TICKER_ALIAS_TO_CANONICAL[canonical.replace("-", "")] = canonical

print("Ticker aliases:", len(TICKER_ALIAS_TO_CANONICAL))

Valid tickers in whitelist: 12511
Ticker aliases: 12824


In [11]:
# ============================================================
# EXTRACT VALID CASHTAG TICKERS FROM POST TEXT
# ============================================================

import re

CASHTAG_REGEX = re.compile(
    r"(?<![A-Za-z0-9])\$([A-Za-z0-9][A-Za-z0-9\.\-]{0,9})(?![A-Za-z0-9])"
)

def extract_valid_cashtag_tickers(text):
    """
    Extracts valid financial tickers from a Bluesky post.
    Only $CASHTAG-style mentions are accepted.
    The extracted symbols are validated against the Nasdaq Trader whitelist.
    """
    if not isinstance(text, str):
        return []

    raw_matches = CASHTAG_REGEX.findall(text)

    valid = set()

    for raw in raw_matches:
        candidate = raw.upper().strip()
        
        if candidate in TICKER_ALIAS_TO_CANONICAL:
            valid.add(TICKER_ALIAS_TO_CANONICAL[candidate])

    return sorted(valid)


# Quick test
test_text = "Watching $AAPL $MSFT $NVDA and $BRK.B today. Also random $LOL."
print(extract_valid_cashtag_tickers(test_text))

['AAPL', 'BRK.B', 'MSFT', 'NVDA']


In [12]:
# ============================================================
# FINANCE-RELATED SEARCH QUERIES
# ============================================================

QUERIES = [
    # General finance
    "stocks",
    "stock market",
    "markets",
    "trading",
    "investing",
    "investors",
    "earnings",
    "earnings season",
    "options",
    "options trading",
    "portfolio",
    "dividends",
    "short squeeze",
    "market rally",
    "market crash",
    "bull market",
    "bear market",

    # Sectors
    "AI stocks",
    "tech stocks",
    "semiconductor stocks",
    "bank stocks",
    "energy stocks",
    "oil stocks",
    "biotech stocks",
    "pharma stocks",
    "EV stocks",
    "defense stocks",

    # ETFs / indices
    "$SPY",
    "$QQQ",
    "$IWM",
    "$DIA",
    "$VTI",
    "$VOO",
    "$TLT",
    "$GLD",
    "$SLV",

    # Mega-cap / frequent finance cashtags
    "$AAPL",
    "$MSFT",
    "$NVDA",
    "$TSLA",
    "$META",
    "$AMZN",
    "$GOOGL",
    "$GOOG",
    "$AMD",
    "$INTC",
    "$NFLX",
    "$AVGO",
    "$ORCL",
    "$PLTR",
    "$SMCI",
    "$COIN",
    "$MSTR",

    # Crypto-related listed equities / common symbols
    "bitcoin stocks",
    "crypto stocks",
    "$BTC",
    "$ETH",
    "$MARA",
    "$RIOT",
    "$CLSK",
    "$HOOD",
]

In [13]:
# ============================================================
# COLLECT POSTS FROM BLUESKY SEARCH
# ============================================================

import time
import pandas as pd
from tqdm import tqdm

def search_posts_for_query(client, query, max_pages=5, limit=100, sleep_seconds=0.5):
    """
    Collects Bluesky posts for a single query using app.bsky.feed.searchPosts.
    """
    posts = []
    cursor = None

    for page in range(max_pages):
        params = {
            "q": query,
            "limit": limit,
        }

        if cursor:
            params["cursor"] = cursor

        try:
            response = client.app.bsky.feed.search_posts(params)
        except Exception as e:
            print(f"Error for query={query}, page={page + 1}: {e}")
            break

        response_posts = getattr(response, "posts", [])

        if not response_posts:
            break

        for post in response_posts:
            record = getattr(post, "record", None)
            text = getattr(record, "text", "") if record else ""
            created_at = getattr(record, "created_at", None) if record else None

            posts.append({
                "query": query,
                "uri": getattr(post, "uri", None),
                "cid": getattr(post, "cid", None),
                "author_handle": getattr(post.author, "handle", None),
                "author_did": getattr(post.author, "did", None),
                "created_at": created_at,
                "text": text,
                "like_count": getattr(post, "like_count", 0),
                "reply_count": getattr(post, "reply_count", 0),
                "repost_count": getattr(post, "repost_count", 0),
                "quote_count": getattr(post, "quote_count", 0),
            })

        cursor = getattr(response, "cursor", None)

        if not cursor:
            break

        time.sleep(sleep_seconds)

    return posts


all_posts = []

for query in tqdm(QUERIES):
    query_posts = search_posts_for_query(
        client=client,
        query=query,
        max_pages=5,
        limit=100,
        sleep_seconds=0.5
    )
    all_posts.extend(query_posts)
    time.sleep(1)

posts_df = pd.DataFrame(all_posts)

if len(posts_df) > 0:
    posts_df = posts_df.drop_duplicates(subset=["uri"]).reset_index(drop=True)

print("Collected posts:", len(posts_df))

display(posts_df.head(10))

  0%|          | 0/61 [00:00<?, ?it/s]

Error for query=stocks, page=4: 


 10%|▉         | 6/61 [00:52<07:26,  8.13s/it]

Error for query=earnings, page=3: 


 25%|██▍       | 15/61 [02:07<05:40,  7.40s/it]

...

 93%|█████████▎| 57/61 [07:02<00:34,  8.69s/it]

Error for query=$MARA, page=5: 


100%|██████████| 61/61 [07:44<00:00,  7.62s/it]

Collected posts: 24766


,query,uri,cid,author_handle,author_did,created_at,text,like_count,reply_count,repost_count,quote_count
0,stocks,at://did:plc:ilqsyryppoq2vgipjp3fnxiw/app.bsky...,bafyreigmwt4idfs3nwiwli7pqe2jsanec42tlounnfnro...,secwatch.bsky.social,did:plc:ilqsyryppoq2vgipjp3fnxiw,2026-06-30T13:07:23.556665Z,🚨 JUST IN: $HCWB\nHCW Biologics regains Nasdaq...,0,0,0,0
1,stocks,at://did:plc:ilqsyryppoq2vgipjp3fnxiw/app.bsky...,bafyreifuu6gsuuxl23yacycktrrgs7m5ban2q7b7cnnot...,secwatch.bsky.social,did:plc:ilqsyryppoq2vgipjp3fnxiw,2026-06-30T13:07:22.402435Z,🚨 JUST IN: $QXO\nQXO reports 99.5%+ tender of ...,0,0,0,0
2,stocks,at://did:plc:gtw57nuw4wt2tqs322knhgim/app.bsky...,bafyreib3paf5vp3ojkgoauwnd73xw2juzqr5mmrdtqvk4...,capitalwisdom.bsky.social,did:plc:gtw57nuw4wt2tqs322knhgim,2026-06-30T13:07:15.980401+00:00,"📈 Capital Wisdom Top Picks — Jun 30, 2026\n\n$...",0,0,0,0
3,stocks,at://did:plc:xhrtfphu4gzqrztkpb3a63qf/app.bsky...,bafyreif355gfx3gqcp6pbyzngywx422y7esnflrfxg6ma...,babygoldie.bsky.social,did:plc:xhrtfphu4gzqrztkpb3a63qf,2026-06-30T13:06:15.262304+00:00,"[3/4] in the future. After verification, the c...",0,1,0,0
4,stocks,at://did:plc:xhrtfphu4gzqrztkpb3a63qf/app.bsky...,bafyreic6sbrvrz47nmpzdktcj36ccjnsdkogpagewbwxk...,babygoldie.bsky.social,did:plc:xhrtfphu4gzqrztkpb3a63qf,2026-06-30T13:05:57.076841+00:00,[3/4] interest rates appear to be overdone. Th...,0,1,0,0
5,stocks,at://did:plc:xhrtfphu4gzqrztkpb3a63qf/app.bsky...,bafyreie3iucy2lea4s5gm37cyuv4qmxkqk7wgeufpngoc...,babygoldie.bsky.social,did:plc:xhrtfphu4gzqrztkpb3a63qf,2026-06-30T13:05:55.738359+00:00,[1/4] 2026-06-30 19:53:22 - [BlackRock downgra...,0,1,0,0
6,stocks,at://did:plc:xhrtfphu4gzqrztkpb3a63qf/app.bsky...,bafyreiayhvaj2ok6d6dzi5ui3kiq4mtdku75yc2izt5hg...,babygoldie.bsky.social,did:plc:xhrtfphu4gzqrztkpb3a63qf,2026-06-30T13:04:29.172633+00:00,"[3/5] characteristics of ""high scores clusteri...",0,1,0,0
7,stocks,at://did:plc:xhrtfphu4gzqrztkpb3a63qf/app.bsky...,bafyreif55gq3ffvddb4saydqfgtdu5ot37ax6hvs4ljfg...,babygoldie.bsky.social,did:plc:xhrtfphu4gzqrztkpb3a63qf,2026-06-30T13:03:50.262565+00:00,"[3/5] characteristics of ""high scores clusteri...",0,1,0,0
8,stocks,at://did:plc:2mmdgkpha4xuzppdqixd7q6g/app.bsky...,bafyreicmfmna74qb4y7l2dgyyrkh7b2hyvtw7qmgozpma...,aistorynews.bsky.social,did:plc:2mmdgkpha4xuzppdqixd7q6g,2026-06-30T13:03:32Z,"AI chipmaker rally lifts Asia markets, stokes ...",0,0,0,0
9,stocks,at://did:plc:brxtbyqsirrv66f3nqqgkvjz/app.bsky...,bafyreib5lpffxshauieg6vwv6wcyay2dpx5qqumh3ekqi...,chelseafinancial.bsky.social,did:plc:brxtbyqsirrv66f3nqqgkvjz,2026-06-30T06:03:27-07:00,#TechStocks #Stocks #USEconomy #2026Economy #I...,0,0,0,0


In [14]:
# ============================================================
# EXTRACT VALID TICKERS FROM COLLECTED POSTS
# ============================================================

posts_df["tickers"] = posts_df["text"].apply(extract_valid_cashtag_tickers)
posts_df["num_tickers"] = posts_df["tickers"].apply(len)

ticker_posts_df = posts_df[posts_df["num_tickers"] >= 2].copy()
ticker_posts_df = ticker_posts_df.reset_index(drop=True)

print("Total posts:", len(posts_df))
print("Posts with at least 1 valid ticker:", (posts_df["num_tickers"] >= 1).sum())
print("Posts with at least 2 valid tickers:", len(ticker_posts_df))

display(
    ticker_posts_df[["created_at", "author_handle", "text", "tickers", "num_tickers"]]
    .head(20)
)

Total posts: 24766
Posts with at least 1 valid ticker: 4048
Posts with at least 2 valid tickers: 1319


,created_at,author_handle,text,tickers,num_tickers
0,2026-06-30T12:55:33.892Z,chrisversace.bsky.social,Apple’s New iPhone Leaks: 8 Key Items Shaping ...,"[AAPL, NKE, NVDA, PLTR, STZ]",5
1,2026-06-30T12:14:21.366Z,bestsidehustle.bsky.social,"Did you watch $BTC, $ETH, $AAPL, $AMZN repeat ...","[AAPL, AMZN, BTC, ETH]",4
2,2026-06-30T12:01:38.214Z,coingapenews.bsky.social,📈 Ark Invest Doubles Down on Crypto Stocks\n\n...,"[BLSH, COIN, CRCL, HOOD]",4
3,2026-06-30T11:01:02.992Z,iamjaredkim.bsky.social,Citi says 80% of Nasdaq longs are losing. That...,"[IWM, USO]",2
4,2026-06-29T18:37:25.000Z,mikezaccardi.bsky.social,Four Big Questions Facing the Stock Market in ...,"[IWM, MAGS, QQQ, SPY]",4
5,2026-06-30T12:53:14.766Z,coingapenews.bsky.social,📉 #MiCA Deadline Tomorrow\n\nMarkets are watch...,"[BTC, ETH, XRP]",3
6,2026-06-30T10:57:43.932Z,tokenaxishq.bsky.social,"Good morning, traders. 📈\nFinal trading day of...","[AMZN, NKE, STZ]",3
7,2026-06-30T03:36:10.330Z,wallstgamenotes.bsky.social,Can Nike Capitalize On Sports Mania? Q4 Earnin...,"[LULU, NKE]",2
8,2026-06-24T13:09:46.000Z,mikezaccardi.bsky.social,Banks $KBE lit heading into mid-year... and ea...,"[JPM, KBE]",2
9,2026-06-07T11:10:01.564518+00:00,kapualabs.bsky.social,When does a beat become a disappointment? This...,"[AMZN, GOOGL]",2


In [15]:
# ============================================================
# SAVE COLLECTED POSTS
# ============================================================

posts_raw_path = OUTPUT_DIR / "bluesky_posts_raw.csv"
posts_tickers_path = OUTPUT_DIR / "bluesky_posts_with_2plus_tickers.csv"

posts_df.to_csv(posts_raw_path, index=False)
ticker_posts_df.to_csv(posts_tickers_path, index=False)

print("Saved raw posts to:", posts_raw_path)
print("Saved ticker posts to:", posts_tickers_path)

Saved raw posts to: /Users/bucco/Downloads/outputs/bluesky_posts_raw.csv
Saved ticker posts to: /Users/bucco/Downloads/outputs/bluesky_posts_with_2plus_tickers.csv


In [16]:
# ============================================================
# LARGE TICKER-SEEDED QUERY LIST
# ============================================================

# Use broad financial symbols because the goal is a large financial asset co-mention network.
candidate_seed_tickers = sorted(financial_symbols)

# Remove ticker formats that are less likely to work well as Bluesky cashtags/search queries.
# We keep simple symbols first: AAPL, MSFT, NVDA, TSLA, etc.
seed_tickers_simple = [
    t for t in candidate_seed_tickers
    if isinstance(t, str)
    and t.isalnum()
    and 1 <= len(t) <= 5
]

# Optional: remove very short symbols because they can be noisy and ambiguous.
# But keep them if they are in the whitelist. For larger coverage, we keep 1-5 length.
print("Candidate simple seed tickers:", len(seed_tickers_simple))

# Start with 2,000. If this works, increase to 5,000 or 10,000.
N_SEED_TICKERS = 2000

seed_tickers = seed_tickers_simple[:N_SEED_TICKERS]
ticker_seed_queries = [f"${ticker}" for ticker in seed_tickers]

print("Ticker-seeded queries:", len(ticker_seed_queries))
print(ticker_seed_queries[:50])

Candidate simple seed tickers: 12354
Ticker-seeded queries: 2000
['$A', '$AA', '$AAA', '$AAAA', '$AAAC', '$AAAD', '$AAAP', '$AAAU', '$AACB', '$AACBR', '$AACBU', '$AACG', '$AACI', '$AACIU', '$AACIW', '$AACO', '$AACOU', '$AACOW', '$AACP', '$AACPR', '$AACPU', '$AACPW', '$AADR', '$AADX', '$AAEQ', '$AAL', '$AALG', '$AAME', '$AAMI', '$AAOG', '$AAOI', '$AAON', '$AAOX', '$AAP', '$AAPB', '$AAPD', '$AAPG', '$AAPL', '$AAPR', '$AAPU', '$AAPW', '$AAPX', '$AAPY', '$AARD', '$AAT', '$AAUA', '$AAUC', '$AAUM', '$AAUS', '$AAVM']


In [19]:
# ============================================================
# LARGE TICKER-SEEDED QUERY LIST — 10,000 TICKERS
# ============================================================

N_SEED_TICKERS = 10000
MAX_PAGES_PER_QUERY = 5
LIMIT_PER_PAGE = 100

# Keep symbols that are more likely to work well as Bluesky cashtags/search terms.
# Example: AAPL, MSFT, NVDA, TSLA, SPY, QQQ.
seed_tickers_simple = [
    t for t in sorted(financial_symbols)
    if isinstance(t, str)
    and t.isalnum()
    and 1 <= len(t) <= 5
]

# Put highly discussed market symbols first.
priority_tickers = [
    "SPY", "QQQ", "IWM", "DIA", "VTI", "VOO", "TLT", "GLD", "SLV",
    "AAPL", "MSFT", "NVDA", "TSLA", "META", "AMZN", "GOOGL", "GOOG",
    "AMD", "INTC", "NFLX", "AVGO", "ORCL", "PLTR", "SMCI", "COIN",
    "MSTR", "MARA", "RIOT", "CLSK", "HOOD", "JPM", "BAC", "GS", "MS",
    "XOM", "CVX", "OXY", "NKE", "DIS", "UBER", "ABNB", "SHOP", "SQ",
    "PYPL", "BABA", "NIO", "RIVN", "LCID", "SOFI", "GME", "AMC"
]

priority_tickers = [
    t for t in priority_tickers
    if t in seed_tickers_simple
]

remaining_tickers = [
    t for t in seed_tickers_simple
    if t not in set(priority_tickers)
]

seed_tickers = priority_tickers + remaining_tickers
seed_tickers = seed_tickers[:N_SEED_TICKERS]

ticker_seed_queries = [f"${ticker}" for ticker in seed_tickers]

print("Available simple seed tickers:", len(seed_tickers_simple))
print("Ticker-seeded queries selected:", len(ticker_seed_queries))
print("Max theoretical requests:", len(ticker_seed_queries) * MAX_PAGES_PER_QUERY)
print("First 50 queries:")
print(ticker_seed_queries[:50])

Available simple seed tickers: 12354
Ticker-seeded queries selected: 10000
Max theoretical requests: 50000
First 50 queries:
['$SPY', '$QQQ', '$IWM', '$DIA', '$VTI', '$VOO', '$TLT', '$GLD', '$SLV', '$AAPL', '$MSFT', '$NVDA', '$TSLA', '$META', '$AMZN', '$GOOGL', '$GOOG', '$AMD', '$INTC', '$NFLX', '$AVGO', '$ORCL', '$PLTR', '$SMCI', '$COIN', '$MSTR', '$MARA', '$RIOT', '$CLSK', '$HOOD', '$JPM', '$BAC', '$GS', '$MS', '$XOM', '$CVX', '$OXY', '$NKE', '$DIS', '$UBER', '$ABNB', '$SHOP', '$PYPL', '$BABA', '$NIO', '$RIVN', '$LCID', '$SOFI', '$GME', '$AMC']


In [20]:
# ============================================================
# LARGE BLUESKY EXTRACTION — 10,000 TICKERS, 5 PAGES EACH
# WITH CHECKPOINTS, COMPLETED QUERY LOG, AND BACKOFF
# ============================================================

import time
import pandas as pd
from tqdm import tqdm
from pathlib import Path

LARGE_POSTS_CHECKPOINT_PATH = OUTPUT_DIR / "bluesky_large_10000_seeded_posts_checkpoint.csv"
COMPLETED_QUERIES_PATH = OUTPUT_DIR / "bluesky_large_10000_completed_queries.txt"
ERROR_LOG_PATH = OUTPUT_DIR / "bluesky_large_10000_error_log.csv"


def load_completed_queries(path):
    if path.exists():
        with open(path, "r") as f:
            return set(line.strip() for line in f if line.strip())
    return set()


def append_completed_query(path, query):
    with open(path, "a") as f:
        f.write(query + "\n")


def load_posts_checkpoint(path):
    if path.exists():
        df = pd.read_csv(path)
        if "uri" in df.columns:
            df = df.drop_duplicates(subset=["uri"]).reset_index(drop=True)
        print("Loaded post checkpoint:", len(df), "unique posts")
        return df
    return pd.DataFrame()


def save_posts_checkpoint(posts_df, path):
    if len(posts_df) == 0:
        return posts_df

    posts_df = posts_df.drop_duplicates(subset=["uri"]).reset_index(drop=True)
    posts_df.to_csv(path, index=False)
    print("Checkpoint saved:", len(posts_df), "unique posts")
    return posts_df


def append_error_log(path, query, page, error_message):
    row = pd.DataFrame([{
        "query": query,
        "page": page,
        "error": str(error_message),
        "timestamp": pd.Timestamp.utcnow().isoformat()
    }])

    if path.exists():
        old = pd.read_csv(path)
        new = pd.concat([old, row], ignore_index=True)
    else:
        new = row

    new.to_csv(path, index=False)


def search_posts_for_query_big(
    client,
    query,
    max_pages=5,
    limit=100,
    sleep_between_pages=0.5,
    max_retries=3
):
    posts = []
    cursor = None

    for page in range(1, max_pages + 1):
        params = {
            "q": query,
            "limit": limit,
        }

        if cursor:
            params["cursor"] = cursor

        success = False

        for attempt in range(1, max_retries + 1):
            try:
                response = client.app.bsky.feed.search_posts(params)
                success = True
                break

            except Exception as e:
                wait_time = 10 * attempt
                print(f"Error query={query}, page={page}, attempt={attempt}: {e}")
                append_error_log(ERROR_LOG_PATH, query, page, e)
                time.sleep(wait_time)

        if not success:
            break

        response_posts = getattr(response, "posts", [])

        if not response_posts:
            break

        for post in response_posts:
            record = getattr(post, "record", None)
            text = getattr(record, "text", "") if record else ""
            created_at = getattr(record, "created_at", None) if record else None

            posts.append({
                "query": query,
                "uri": getattr(post, "uri", None),
                "cid": getattr(post, "cid", None),
                "author_handle": getattr(post.author, "handle", None),
                "author_did": getattr(post.author, "did", None),
                "created_at": created_at,
                "text": text,
                "like_count": getattr(post, "like_count", 0),
                "reply_count": getattr(post, "reply_count", 0),
                "repost_count": getattr(post, "repost_count", 0),
                "quote_count": getattr(post, "quote_count", 0),
            })

        cursor = getattr(response, "cursor", None)

        if not cursor:
            break

        time.sleep(sleep_between_pages)

    return posts


# Load existing progress if present
completed_queries = load_completed_queries(COMPLETED_QUERIES_PATH)
large_posts_df = load_posts_checkpoint(LARGE_POSTS_CHECKPOINT_PATH)

queries_to_run = [
    q for q in ticker_seed_queries
    if q not in completed_queries
]

print("Total target queries:", len(ticker_seed_queries))
print("Already completed queries:", len(completed_queries))
print("Remaining queries:", len(queries_to_run))
print("Current checkpoint posts:", len(large_posts_df))

SAVE_EVERY_N_QUERIES = 25
SLEEP_BETWEEN_QUERIES = 0.8
LONG_PAUSE_EVERY_N_QUERIES = 250
LONG_PAUSE_SECONDS = 30

new_rows_buffer = []

for i, query in enumerate(tqdm(queries_to_run), start=1):
    query_posts = search_posts_for_query_big(
        client=client,
        query=query,
        max_pages=MAX_PAGES_PER_QUERY,
        limit=LIMIT_PER_PAGE,
        sleep_between_pages=0.5,
        max_retries=3
    )

    if len(query_posts) > 0:
        new_rows_buffer.extend(query_posts)

    append_completed_query(COMPLETED_QUERIES_PATH, query)

    if i % SAVE_EVERY_N_QUERIES == 0:
        if len(new_rows_buffer) > 0:
            buffer_df = pd.DataFrame(new_rows_buffer)
            large_posts_df = pd.concat(
                [large_posts_df, buffer_df],
                ignore_index=True
            )
            new_rows_buffer = []

        large_posts_df = save_posts_checkpoint(
            large_posts_df,
            LARGE_POSTS_CHECKPOINT_PATH
        )

        completed_now = len(load_completed_queries(COMPLETED_QUERIES_PATH))
        print(
            f"Progress: {completed_now}/{len(ticker_seed_queries)} queries completed | "
            f"{len(large_posts_df)} unique posts saved"
        )

    if i % LONG_PAUSE_EVERY_N_QUERIES == 0:
        print(f"Long pause after {i} queries...")
        time.sleep(LONG_PAUSE_SECONDS)

    time.sleep(SLEEP_BETWEEN_QUERIES)


# Final save
if len(new_rows_buffer) > 0:
    buffer_df = pd.DataFrame(new_rows_buffer)
    large_posts_df = pd.concat(
        [large_posts_df, buffer_df],
        ignore_index=True
    )

large_posts_df = save_posts_checkpoint(
    large_posts_df,
    LARGE_POSTS_CHECKPOINT_PATH
)

completed_queries = load_completed_queries(COMPLETED_QUERIES_PATH)

print("DONE.")
print("Completed queries:", len(completed_queries))
print("Large unique posts:", len(large_posts_df))
print("Saved posts checkpoint to:", LARGE_POSTS_CHECKPOINT_PATH)
print("Saved completed queries to:", COMPLETED_QUERIES_PATH)

display(large_posts_df.head(10))

Total target queries: 10000
Already completed queries: 0
Remaining queries: 10000
Current checkpoint posts: 0


  0%|          | 19/10000 [02:18<18:05:40,  6.53s/it]

Error query=$NFLX, page=3, attempt=1: 


/var/folders/pw/y3ztzc9n5ns3pn2xk2j59d2w0000gp/T/ipykernel_7530/2195544028.py:53: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  "timestamp": pd.Timestamp.utcnow().isoformat()
  0%|          | 20/10000 [02:39<30:36:25, 11.04s/it]

Error query=$AVGO, page=2, attempt=1: 


...

100%|█████████▉| 9999/10000 [16:04:53<00:01,  1.44s/it]

Checkpoint saved: 2487835 unique posts
Progress: 10000/10000 queries completed | 2487835 unique posts saved
Long pause after 10000 queries...


100%|██████████| 10000/10000 [16:05:42<00:00,  5.79s/it]


Checkpoint saved: 2487835 unique posts
DONE.
Completed queries: 10000
Large unique posts: 2487835
Saved posts checkpoint to: /Users/bucco/Downloads/outputs/bluesky_large_10000_seeded_posts_checkpoint.csv
Saved completed queries to: /Users/bucco/Downloads/outputs/bluesky_large_10000_completed_queries.txt


,query,uri,cid,author_handle,author_did,created_at,text,like_count,reply_count,repost_count,quote_count
0,$SPY,at://did:plc:7vcmsr34a4fmn33syokw62ir/app.bsky...,bafyreihetpfrvpdurm7k7kpd66vmpbg43iwkm3dzrqxy2...,ericcalkins.bsky.social,did:plc:7vcmsr34a4fmn33syokw62ir,2026-06-30T13:20:54.807Z,Related: suppose one had the choice of three I...,0,0,0,0
1,$SPY,at://did:plc:z6rujpf4u56jfie7aqic2nfg/app.bsky...,bafyreiffxx2fpkigk7xrfb3thwk62vdd7ctltviivxg2p...,carlquintanilla.bsky.social,did:plc:z6rujpf4u56jfie7aqic2nfg,2026-06-30T13:20:48.581Z,"OPCO DESK: "".. For only second time since 2010...",5,0,1,0
2,$SPY,at://did:plc:kmc3trloj2jezhby2k2nadkr/app.bsky...,bafyreigrenhoxlc2wf2d7dhuvttfjpvthxps5z55vm2n3...,magnoliara.bsky.social,did:plc:kmc3trloj2jezhby2k2nadkr,2026-06-30T13:20:42.688Z,KSE: btw im a spy who infiltrated the gov\nBro...,0,0,0,0
3,$SPY,at://did:plc:47ukz5svch7g2jmqu674y7b5/app.bsky...,bafyreicfkg2ftkwhzeooglrqh3ppxqiswflk55oqaacmn...,eu-gb.bsky.social,did:plc:47ukz5svch7g2jmqu674y7b5,2026-06-30T13:15:15+00:00,Nuclear weapons storage in spotlight as US pla...,0,0,0,0
4,$SPY,at://did:plc:nnfjy7mgugylrbiasa7pqidb/app.bsky...,bafyreibgfnkxwmfxscf3av65optvyvxjrczhcyce7oqwz...,notimeforwaifus.bsky.social,did:plc:nnfjy7mgugylrbiasa7pqidb,2026-06-30T13:15:08.449Z,"Yor, your beauty and elegance are completely i...",3,0,0,0
5,$SPY,at://did:plc:mbt5u6e4b77dgihfhvlvpkxt/app.bsky...,bafyreigb6qk5a4lgy6noetsksfamfs3xpm2wos5hvy6by...,teawhalem.bsky.social,did:plc:mbt5u6e4b77dgihfhvlvpkxt,2026-06-30T13:15:03.883Z,The Last Act of an Unusual Spy Chief\nopen.sub...,0,0,0,1
6,$SPY,at://did:plc:k6orgxkypgzp3dmkzyu37qzk/app.bsky...,bafyreieqcvoytwsaxywyqsg3bfmfv5pklr27mvbqw4ijh...,numbcat9.bsky.social,did:plc:k6orgxkypgzp3dmkzyu37qzk,2026-06-30T13:13:32.674Z,He literally let the head of Russia’s spy agen...,0,0,1,0
7,$SPY,at://did:plc:bfrqponecue5czek3awk6ydd/app.bsky...,bafyreid2plus4xavabal245rs7mr254wlscn7lfids3bq...,pixelwelten.eurosky.social,did:plc:bfrqponecue5czek3awk6ydd,2026-06-30T13:06:58.301Z,"Pokemon Go kloppen, aber mit Rayban und Meta C...",1,0,0,0
8,$SPY,at://did:plc:nnfjy7mgugylrbiasa7pqidb/app.bsky...,bafyreiefsq3fk3eip3qn42puf2djituetb6arcscnn74v...,notimeforwaifus.bsky.social,did:plc:nnfjy7mgugylrbiasa7pqidb,2026-06-30T13:02:58.952Z,"The most intimate, mouth-watering view of Yor....",7,0,0,0
9,$SPY,at://did:plc:q2bofj35scvkbetpou2ulrt4/app.bsky...,bafyreia2vnc7pse2l6b3grlyqzt6vhbrpy2uinl52up3z...,jacobschneider.bsky.social,did:plc:q2bofj35scvkbetpou2ulrt4,2026-06-30T13:02:35.993Z,I spy something perfect,1,0,1,0


In [21]:
# ============================================================
# MERGE ORIGINAL FINANCE-QUERY RUN WITH LARGE TICKER-SEEDED RUN
# ============================================================

large_posts_df = pd.read_csv(LARGE_POSTS_CHECKPOINT_PATH)

combined_posts_df = pd.concat(
    [posts_df, large_posts_df],
    ignore_index=True
)

combined_posts_df = combined_posts_df.drop_duplicates(subset=["uri"]).reset_index(drop=True)

combined_posts_path = OUTPUT_DIR / "bluesky_posts_combined_10000_seeded.csv"
combined_posts_df.to_csv(combined_posts_path, index=False)

print("Original finance-query posts:", len(posts_df))
print("Large ticker-seeded posts:", len(large_posts_df))
print("Combined unique posts:", len(combined_posts_df))
print("Saved combined posts to:", combined_posts_path)

display(combined_posts_df.head(10))

Original finance-query posts: 24766
Large ticker-seeded posts: 2487835
Combined unique posts: 2498033
Saved combined posts to: /Users/bucco/Downloads/outputs/bluesky_posts_combined_10000_seeded.csv


,query,uri,cid,author_handle,author_did,created_at,text,like_count,reply_count,repost_count,quote_count,tickers,num_tickers
0,stocks,at://did:plc:ilqsyryppoq2vgipjp3fnxiw/app.bsky...,bafyreigmwt4idfs3nwiwli7pqe2jsanec42tlounnfnro...,secwatch.bsky.social,did:plc:ilqsyryppoq2vgipjp3fnxiw,2026-06-30T13:07:23.556665Z,🚨 JUST IN: $HCWB\nHCW Biologics regains Nasdaq...,0,0,0,0,[HCWB],1.0
1,stocks,at://did:plc:ilqsyryppoq2vgipjp3fnxiw/app.bsky...,bafyreifuu6gsuuxl23yacycktrrgs7m5ban2q7b7cnnot...,secwatch.bsky.social,did:plc:ilqsyryppoq2vgipjp3fnxiw,2026-06-30T13:07:22.402435Z,🚨 JUST IN: $QXO\nQXO reports 99.5%+ tender of ...,0,0,0,0,[QXO],1.0
2,stocks,at://did:plc:gtw57nuw4wt2tqs322knhgim/app.bsky...,bafyreib3paf5vp3ojkgoauwnd73xw2juzqr5mmrdtqvk4...,capitalwisdom.bsky.social,did:plc:gtw57nuw4wt2tqs322knhgim,2026-06-30T13:07:15.980401+00:00,"📈 Capital Wisdom Top Picks — Jun 30, 2026\n\n$...",0,0,0,0,[PEP],1.0
3,stocks,at://did:plc:xhrtfphu4gzqrztkpb3a63qf/app.bsky...,bafyreif355gfx3gqcp6pbyzngywx422y7esnflrfxg6ma...,babygoldie.bsky.social,did:plc:xhrtfphu4gzqrztkpb3a63qf,2026-06-30T13:06:15.262304+00:00,"[3/4] in the future. After verification, the c...",0,1,0,0,[],0.0
4,stocks,at://did:plc:xhrtfphu4gzqrztkpb3a63qf/app.bsky...,bafyreic6sbrvrz47nmpzdktcj36ccjnsdkogpagewbwxk...,babygoldie.bsky.social,did:plc:xhrtfphu4gzqrztkpb3a63qf,2026-06-30T13:05:57.076841+00:00,[3/4] interest rates appear to be overdone. Th...,0,1,0,0,[],0.0
5,stocks,at://did:plc:xhrtfphu4gzqrztkpb3a63qf/app.bsky...,bafyreie3iucy2lea4s5gm37cyuv4qmxkqk7wgeufpngoc...,babygoldie.bsky.social,did:plc:xhrtfphu4gzqrztkpb3a63qf,2026-06-30T13:05:55.738359+00:00,[1/4] 2026-06-30 19:53:22 - [BlackRock downgra...,0,1,0,0,[],0.0
6,stocks,at://did:plc:xhrtfphu4gzqrztkpb3a63qf/app.bsky...,bafyreiayhvaj2ok6d6dzi5ui3kiq4mtdku75yc2izt5hg...,babygoldie.bsky.social,did:plc:xhrtfphu4gzqrztkpb3a63qf,2026-06-30T13:04:29.172633+00:00,"[3/5] characteristics of ""high scores clusteri...",0,1,0,0,[],0.0
7,stocks,at://did:plc:xhrtfphu4gzqrztkpb3a63qf/app.bsky...,bafyreif55gq3ffvddb4saydqfgtdu5ot37ax6hvs4ljfg...,babygoldie.bsky.social,did:plc:xhrtfphu4gzqrztkpb3a63qf,2026-06-30T13:03:50.262565+00:00,"[3/5] characteristics of ""high scores clusteri...",0,1,0,0,[],0.0
8,stocks,at://did:plc:2mmdgkpha4xuzppdqixd7q6g/app.bsky...,bafyreicmfmna74qb4y7l2dgyyrkh7b2hyvtw7qmgozpma...,aistorynews.bsky.social,did:plc:2mmdgkpha4xuzppdqixd7q6g,2026-06-30T13:03:32Z,"AI chipmaker rally lifts Asia markets, stokes ...",0,0,0,0,[],0.0
9,stocks,at://did:plc:brxtbyqsirrv66f3nqqgkvjz/app.bsky...,bafyreib5lpffxshauieg6vwv6wcyay2dpx5qqumh3ekqi...,chelseafinancial.bsky.social,did:plc:brxtbyqsirrv66f3nqqgkvjz,2026-06-30T06:03:27-07:00,#TechStocks #Stocks #USEconomy #2026Economy #I...,0,0,0,0,[],0.0


In [22]:
# ============================================================
# EXTRACT VALID TICKERS FROM LARGE COMBINED DATASET
# ============================================================

combined_posts_df["tickers"] = combined_posts_df["text"].apply(extract_valid_cashtag_tickers)
combined_posts_df["num_tickers"] = combined_posts_df["tickers"].apply(len)

combined_posts_with_1plus_ticker_df = combined_posts_df[
    combined_posts_df["num_tickers"] >= 1
].copy().reset_index(drop=True)

combined_ticker_posts_df = combined_posts_df[
    combined_posts_df["num_tickers"] >= 2
].copy().reset_index(drop=True)

unique_observed_tickers = sorted({
    ticker
    for tickers in combined_posts_df["tickers"]
    for ticker in tickers
})

print("Combined posts:", len(combined_posts_df))
print("Posts with at least 1 valid ticker:", len(combined_posts_with_1plus_ticker_df))
print("Posts with at least 2 valid tickers:", len(combined_ticker_posts_df))
print("Unique observed valid tickers:", len(unique_observed_tickers))

combined_posts_with_1plus_ticker_path = OUTPUT_DIR / "bluesky_posts_combined_10000_with_1plus_ticker.csv"
combined_ticker_posts_path = OUTPUT_DIR / "bluesky_posts_combined_10000_with_2plus_tickers.csv"
observed_tickers_path = OUTPUT_DIR / "bluesky_observed_valid_tickers_10000.txt"

combined_posts_with_1plus_ticker_df.to_csv(combined_posts_with_1plus_ticker_path, index=False)
combined_ticker_posts_df.to_csv(combined_ticker_posts_path, index=False)

with open(observed_tickers_path, "w") as f:
    for ticker in unique_observed_tickers:
        f.write(ticker + "\n")

print("Saved 1+ ticker posts to:", combined_posts_with_1plus_ticker_path)
print("Saved 2+ ticker posts to:", combined_ticker_posts_path)
print("Saved observed tickers to:", observed_tickers_path)

display(
    combined_ticker_posts_df[
        ["created_at", "author_handle", "text", "tickers", "num_tickers"]
    ].head(20)
)

Combined posts: 2498033
Posts with at least 1 valid ticker: 116461
Posts with at least 2 valid tickers: 25229
Unique observed valid tickers: 7642
Saved 1+ ticker posts to: /Users/bucco/Downloads/outputs/bluesky_posts_combined_10000_with_1plus_ticker.csv
Saved 2+ ticker posts to: /Users/bucco/Downloads/outputs/bluesky_posts_combined_10000_with_2plus_tickers.csv
Saved observed tickers to: /Users/bucco/Downloads/outputs/bluesky_observed_valid_tickers_10000.txt


,created_at,author_handle,text,tickers,num_tickers
0,2026-06-30T12:55:33.892Z,chrisversace.bsky.social,Apple’s New iPhone Leaks: 8 Key Items Shaping ...,"[AAPL, NKE, NVDA, PLTR, STZ]",5
1,2026-06-30T12:14:21.366Z,bestsidehustle.bsky.social,"Did you watch $BTC, $ETH, $AAPL, $AMZN repeat ...","[AAPL, AMZN, BTC, ETH]",4
2,2026-06-30T12:01:38.214Z,coingapenews.bsky.social,📈 Ark Invest Doubles Down on Crypto Stocks\n\n...,"[BLSH, COIN, CRCL, HOOD]",4
3,2026-06-30T11:01:02.992Z,iamjaredkim.bsky.social,Citi says 80% of Nasdaq longs are losing. That...,"[IWM, USO]",2
4,2026-06-29T18:37:25.000Z,mikezaccardi.bsky.social,Four Big Questions Facing the Stock Market in ...,"[IWM, MAGS, QQQ, SPY]",4
5,2026-06-30T12:53:14.766Z,coingapenews.bsky.social,📉 #MiCA Deadline Tomorrow\n\nMarkets are watch...,"[BTC, ETH, XRP]",3
6,2026-06-30T10:57:43.932Z,tokenaxishq.bsky.social,"Good morning, traders. 📈\nFinal trading day of...","[AMZN, NKE, STZ]",3
7,2026-06-30T03:36:10.330Z,wallstgamenotes.bsky.social,Can Nike Capitalize On Sports Mania? Q4 Earnin...,"[LULU, NKE]",2
8,2026-06-24T13:09:46.000Z,mikezaccardi.bsky.social,Banks $KBE lit heading into mid-year... and ea...,"[JPM, KBE]",2
9,2026-06-07T11:10:01.564518+00:00,kapualabs.bsky.social,When does a beat become a disappointment? This...,"[AMZN, GOOGL]",2


In [23]:
# ============================================================
# BUILD LARGE TICKER CO-MENTION GRAPH
# ============================================================

from itertools import combinations
from collections import Counter
import pandas as pd
import networkx as nx

large_edge_counter = Counter()

for tickers in combined_ticker_posts_df["tickers"]:
    unique_tickers = sorted(set(tickers))

    if len(unique_tickers) >= 2:
        for source, target in combinations(unique_tickers, 2):
            large_edge_counter[(source, target)] += 1

large_edges_weighted_df = pd.DataFrame(
    [
        {
            "source": source,
            "target": target,
            "weight": weight
        }
        for (source, target), weight in large_edge_counter.items()
    ]
)

large_edges_weighted_df = large_edges_weighted_df.sort_values(
    ["weight", "source", "target"],
    ascending=[False, True, True]
).reset_index(drop=True)

large_edges_unweighted_df = large_edges_weighted_df[["source", "target"]].copy()
large_edges_unweighted_df["weight"] = 1

G_large_weighted = nx.Graph()

for _, row in large_edges_weighted_df.iterrows():
    G_large_weighted.add_edge(
        row["source"],
        row["target"],
        weight=int(row["weight"])
    )

G_large_unweighted = nx.Graph()

for _, row in large_edges_unweighted_df.iterrows():
    G_large_unweighted.add_edge(
        row["source"],
        row["target"],
        weight=1
    )

print("Large weighted graph")
print("Nodes:", G_large_weighted.number_of_nodes())
print("Edges:", G_large_weighted.number_of_edges())

print("\nLarge unweighted graph")
print("Nodes:", G_large_unweighted.number_of_nodes())
print("Edges:", G_large_unweighted.number_of_edges())

display(large_edges_weighted_df.head(30))

Large weighted graph
Nodes: 6243
Edges: 462904

Large unweighted graph
Nodes: 6243
Edges: 462904


,source,target,weight
0,QQQ,SPY,2824
1,DIA,SPY,2053
2,DIA,QQQ,2052
3,BTC,ETH,912
4,NVDA,TSLA,655
5,NVDA,SPY,579
6,AAPL,NVDA,518
7,SPY,TSLA,517
8,AAPL,TSLA,497
9,NVDA,QQQ,487


In [24]:
# ============================================================
# SAVE LARGE GRAPH OUTPUTS FOR REPORT AND GEPHI
# ============================================================

if G_large_unweighted.number_of_nodes() > 0:
    connected_components = list(nx.connected_components(G_large_unweighted))
    largest_component = max(connected_components, key=len)
else:
    connected_components = []
    largest_component = set()

large_basic_stats = {
    "num_posts_combined": len(combined_posts_df),
    "num_posts_with_at_least_1_valid_ticker": len(combined_posts_with_1plus_ticker_df),
    "num_posts_with_at_least_2_valid_tickers": len(combined_ticker_posts_df),
    "unique_observed_valid_tickers": len(unique_observed_tickers),
    "num_nodes": G_large_unweighted.number_of_nodes(),
    "num_edges": G_large_unweighted.number_of_edges(),
    "density": nx.density(G_large_unweighted),
    "num_connected_components": nx.number_connected_components(G_large_unweighted),
    "largest_component_size": len(largest_component),
    "largest_component_share": (
        len(largest_component) / G_large_unweighted.number_of_nodes()
        if G_large_unweighted.number_of_nodes() > 0 else 0
    ),
    "average_degree": (
        sum(dict(G_large_unweighted.degree()).values()) / G_large_unweighted.number_of_nodes()
        if G_large_unweighted.number_of_nodes() > 0 else 0
    )
}

large_basic_stats_df = pd.DataFrame([large_basic_stats])

display(large_basic_stats_df)

# Save tables
large_edges_weighted_path = OUTPUT_DIR / "bluesky_large_edges_weighted_10000.csv"
large_edges_unweighted_path = OUTPUT_DIR / "bluesky_large_edges_unweighted_10000.csv"
large_basic_stats_path = OUTPUT_DIR / "bluesky_large_basic_stats_10000.csv"

large_edges_weighted_df.to_csv(large_edges_weighted_path, index=False)
large_edges_unweighted_df.to_csv(large_edges_unweighted_path, index=False)
large_basic_stats_df.to_csv(large_basic_stats_path, index=False)

# Save graph files
large_graphml_unweighted_path = OUTPUT_DIR / "bluesky_large_network_unweighted_10000.graphml"
large_gexf_unweighted_path = OUTPUT_DIR / "bluesky_large_network_unweighted_10000.gexf"
large_graphml_weighted_path = OUTPUT_DIR / "bluesky_large_network_weighted_10000.graphml"
large_gexf_weighted_path = OUTPUT_DIR / "bluesky_large_network_weighted_10000.gexf"

nx.write_graphml(G_large_unweighted, large_graphml_unweighted_path)
nx.write_gexf(G_large_unweighted, large_gexf_unweighted_path)

nx.write_graphml(G_large_weighted, large_graphml_weighted_path)
nx.write_gexf(G_large_weighted, large_gexf_weighted_path)

print("Saved weighted edges to:", large_edges_weighted_path)
print("Saved unweighted edges to:", large_edges_unweighted_path)
print("Saved stats to:", large_basic_stats_path)
print("Saved unweighted GraphML to:", large_graphml_unweighted_path)
print("Saved unweighted GEXF to:", large_gexf_unweighted_path)
print("Saved weighted GraphML to:", large_graphml_weighted_path)
print("Saved weighted GEXF to:", large_gexf_weighted_path)

,num_posts_combined,num_posts_with_at_least_1_valid_ticker,num_posts_with_at_least_2_valid_tickers,unique_observed_valid_tickers,num_nodes,num_edges,density,num_connected_components,largest_component_size,largest_component_share,average_degree
0,2498033,116461,25229,7642,6243,462904,0.023758,16,6211,0.994874,148.295371


Saved weighted edges to: /Users/bucco/Downloads/outputs/bluesky_large_edges_weighted_10000.csv
Saved unweighted edges to: /Users/bucco/Downloads/outputs/bluesky_large_edges_unweighted_10000.csv
Saved stats to: /Users/bucco/Downloads/outputs/bluesky_large_basic_stats_10000.csv
Saved unweighted GraphML to: /Users/bucco/Downloads/outputs/bluesky_large_network_unweighted_10000.graphml
Saved unweighted GEXF to: /Users/bucco/Downloads/outputs/bluesky_large_network_unweighted_10000.gexf
Saved weighted GraphML to: /Users/bucco/Downloads/outputs/bluesky_large_network_weighted_10000.graphml
Saved weighted GEXF to: /Users/bucco/Downloads/outputs/bluesky_large_network_weighted_10000.gexf
